In [19]:
import time
import sys
import os
from typing import TypedDict, List
from openai import OpenAI
import gradio as gr

In [6]:
# Config

# LLM clients
openai = OpenAI()

# LLM models
OPEN_AI_MODEL = "gpt-4o-mini"

# Instructions
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get. \
The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

In [24]:
class ChatMessage(TypedDict):
    role: str
    content: str


def chat(message: str, history: List[ChatMessage]):
    messages = [{"role": "system", "content": system_message}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": message})

    stream = openai.chat.completions.create(
        model=OPEN_AI_MODEL, messages=messages, stream=True
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


history = [
    {
        "role": "user", 
        "content": "I am thinking to buy something...."},
    {
        "role": "assistant",
        "content": "That sounds great! We have some fantastic items on sale right now. For instance, \
            our hats are 60% off! They could be a stylish addition to your wardrobe. \
                We also have a variety of other clothing items at 50% off. \
                      Is there something specific you're looking for, \
                        or would you like to explore some of our sale items?",
    },
]
previous = ""

for token in chat("I change my mind, what about the belts. Do you have some?", history=history):
    # Erase previous output
    sys.stdout.write("\r" + " " * len(previous))
    sys.stdout.flush()

    # Write new output
    sys.stdout.write("\r" + token)
    sys.stdout.flush()

    previous = token
    time.sleep(0.01)  # optional for typing effect

I’m sorry, but we don’t carry belts. However, we have a wonderful selection of other items on sale! Our hats are 60% off right now, and many other clothing items are 50% off. You might find something you love in our sale section! Would you like to check out the hats or perhaps some clothing?

In [25]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
